<a href="https://colab.research.google.com/github/Steph-business/Tech_Talent_Accelerator/blob/main/Week6_day1_dalychallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Défi Quotidien: Mécanisme d'Attention Personnalisé & Classification de Spam SMS

Bienvenue dans le cahier guidé pour le défi quotidien *Mécanisme d'Attention Personnalisé & Spam SMS*. Les cellules étiquetées **PRE-REMPLIES** sont prêtes à être exécutées telles quelles. Les cellules étiquetées **À Faire** vous demandent de remplacer le code ou le texte de l'espace réservé par votre propre travail avant d'exécuter le cahier.

## Pourquoi faisons-nous cela ?
Les systèmes NLP modernes reposent sur l'attention. En créant votre propre bloc d'attention et en le comparant à un classificateur GPT-2 pré-entraîné, vous démystifierez comment les flux de requête/clé/valeur façonnent les prédictions en aval sur un véritable ensemble de données de spam SMS.

![Image](https://github.com/user-attachments/assets/bc4d5315-983b-4fc1-9011-25fa743bb25f)

## Objectifs d'apprentissage
- Implémenter une couche d'attention personnalisée à produit scalaire mis à l'échelle à partir de zéro.
- Expliquer les rôles respectifs des requêtes, des clés et des valeurs.
- Affiner GPT-2 pour la classification binaire de spam et le comparer à un modèle personnalisé.
- Évaluer les deux systèmes avec la précision, le rappel et le score F1.
- Réfléchir aux compromis entre les modèles basés sur les transformeurs et les modèles d'attention légers.

> **Point d'apprentissage**
> Travaillez chaque partie séquentiellement. Remplacez chaque marqueur `# TODO:` avant d'exécuter la cellule afin que les étapes en aval (tokenisation, entraînement, évaluation) reçoivent les entrées attendues.

# Partie 1: Configuration et Chargement des Données
Comme sur la plateforme, commencez par installer les dépendances, importer les modules d'aide et découper l'ensemble de données SMS en 4 000 lignes d'entraînement et 1 000 lignes de validation.

**PRÉ-REMPLI: exécuter une fois**
Installe les bibliothèques nécessaires à ce défi.

In [1]:
%pip install --quiet datasets evaluate transformers[sentencepiece]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


**À Faire (code)**
Importez pandas ainsi que les utilitaires de jeu de données exactement comme indiqué dans les instructions de la plateforme.

In [2]:
import pandas as pd
from datasets import Dataset
from datasets import load_dataset

**À Faire (code)**
Chargez le fichier Parquet UCI SMS Spam, convertissez-le en un jeu de données Hugging Face, puis créez des découpes de 4 000 / 1 000 comme décrit dans l'énoncé.

In [3]:
# TODO: load and inspect the SMS Spam dataset
DATA_PATH = 'hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet'
df = pd.read_parquet(DATA_PATH)  # load the parquet dataset into a pandas DataFrame
hf_dataset = Dataset.from_pandas(df)  # convert the DataFrame into a Hugging Face Dataset

TRAIN_START = 0
TRAIN_END = 4000  # TODO: use 4,000 samples for training
VAL_START = 4000  # TODO: begin validation split at 4,000
VAL_END = 5000    # TODO: stop validation split at 5,000

if None in (TRAIN_END, VAL_START, VAL_END):
    raise ValueError('Set TRAIN_END, VAL_START, and VAL_END according to the instructions.')

train_ds = hf_dataset.select(range(TRAIN_START, TRAIN_END))
val_ds = hf_dataset.select(range(VAL_START, VAL_END))
display(df.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


,sms,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...\n,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


In [4]:
import pandas as pd
from datasets import Dataset

# Re-chargement forcé pour s'assurer que les variables existent en mémoire
DATA_PATH = 'hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet'
df = pd.read_parquet(DATA_PATH)
hf_dataset = Dataset.from_pandas(df)

train_ds = hf_dataset.select(range(0, 4000))
val_ds = hf_dataset.select(range(4000, 5000))

print(f'Variables initialisées : train_ds ({len(train_ds)} lignes), val_ds ({len(val_ds)} lignes)')

Variables initialisées : train_ds (4000 lignes), val_ds (1000 lignes)


# Partie 2: Configuration de la Tokenisation
Initialisez le tokenizer GPT-2, définissez un jeton de remplissage et préparez la tokenisation par lots pour les deux découpes.

> **Point d'apprentissage**
> GPT-2 ne définit pas de jeton de remplissage. Réutiliser le jeton EOS permet de maintenir les entrées alignées avec la façon dont le modèle a été pré-entraîné.

In [5]:
# TODO: initialize the tokenizer and padding behavior
from transformers import GPT2Tokenizer

MODEL_NAME = "gpt2"  # TODO: set to 'gpt2', you can also try 'gpt2-medium' or 'gpt2-large'
if MODEL_NAME is None:
    raise ValueError("Set MODEL_NAME to the pretrained checkpoint (e.g., 'gpt2').")

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # TODO: verify pad token is mapped to eos


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [6]:
# TODO: complete the tokenization function
TEXT_COLUMN = "sms"
PADDING_STRATEGY = "max_length"
TRUNCATION_FLAG = True
MAX_SEQ_LEN = 64

for setting in (TEXT_COLUMN, PADDING_STRATEGY, TRUNCATION_FLAG, MAX_SEQ_LEN):
    if setting is None:
        raise ValueError('Complete TEXT_COLUMN, PADDING_STRATEGY, TRUNCATION_FLAG, and MAX_SEQ_LEN.')

def tokenize_fn(examples):
    return tokenizer(
        examples[TEXT_COLUMN],
        padding=PADDING_STRATEGY,
        truncation=TRUNCATION_FLAG,
        max_length=MAX_SEQ_LEN,
    )

# Cette fois-ci, les variables train_ds et val_ds sont bien définies
train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

# Partie 3: Classificateur GPT-2 Pré-entraîné
Chargez GPT-2 avec une tête de classification adaptée à la détection binaire de spam.

In [7]:
# TODO: instantiate GPT-2 for sequence classification
import torch
from transformers import GPT2ForSequenceClassification

NUM_LABELS = 2  # TODO: set to 2 for spam vs. ham because this is binary classification
if NUM_LABELS is None:
    raise ValueError('Set NUM_LABELS to 2 for binary classification.')

model = GPT2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    pad_token_id=tokenizer.eos_token_id,  # TODO: confirm pad token id so training does not error out
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Partie 4: Implémentation de l'Attention Personnalisée
Construisez la couche d'attention simple, le classificateur et le pipeline de données pour le modèle à partir de zéro.

> **Point d'apprentissage**
> Mettre à l'échelle les produits scalaires par $1/\sqrt{d_k}$ maintient les gradients stables et empêche le softmax de s'effondrer lorsque les plongements augmentent. Cette opération est cruciale pour l'entraînement des modèles d'attention profonds.

In [8]:
# TODO: implement the Attention layer
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.scale = embed_dim ** -0.5  # Scale factor 1/sqrt(d_k)

    def forward(self, query, key, value, mask=None):
        # Dot product query and key (transposed)
        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # Apply softmax over the last dimension
        attn = F.softmax(scores, dim=-1)

        # Multiply attention weights by values
        return torch.matmul(attn, value), attn


class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attn = Attention(embed_dim)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embed = self.embedding(x)

        # Self-attention: query, key, and value are all the same embedding
        attn_output, _ = self.attn(embed, embed, embed)

        # Global average pooling over the sequence dimension (dim 1)
        pooled = attn_output.mean(dim=1)

        return self.fc(pooled)

> **Point d'apprentissage**
> Tokenisez une fois et réutilisez la même limite de 64 jetons afin que les deux modèles reçoivent des fenêtres de contexte comparables.

In [9]:
# TODO: preprocess datasets for the custom attention model
ATTN_TEXT_COLUMN = 'sms'  # TODO: set to 'sms'
ATTN_MAX_LEN = 64      # TODO: set to 64
if ATTN_TEXT_COLUMN is None or ATTN_MAX_LEN is None:
    raise ValueError('Complete ATTN_TEXT_COLUMN and ATTN_MAX_LEN.')


def preprocess_for_attention(example):
    tokens = tokenizer.encode(
        example[ATTN_TEXT_COLUMN],
        max_length=ATTN_MAX_LEN,
        truncation=True,
        padding='max_length',
    )
    return {'input_ids': tokens, 'label': example['label']}


train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn = val_ds.map(preprocess_for_attention)


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [10]:
# TODO: create PyTorch DataLoaders
class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'label': torch.tensor(item['label'], dtype=torch.long),
        }


TRAIN_DATA_FOR_LOADER = train_ds_attn  # TODO: set to train_ds_attn
VAL_DATA_FOR_LOADER = val_ds_attn    # TODO: set to val_ds_attn
if TRAIN_DATA_FOR_LOADER is None or VAL_DATA_FOR_LOADER is None:
    raise ValueError('Assign TRAIN_DATA_FOR_LOADER and VAL_DATA_FOR_LOADER before creating loaders.')


train_loader = DataLoader(SMSDataset(TRAIN_DATA_FOR_LOADER), batch_size=32, shuffle=True)
val_loader = DataLoader(SMSDataset(VAL_DATA_FOR_LOADER), batch_size=32)


In [11]:
# TODO: train the custom attention classifier
vocab_size = len(tokenizer)  # TODO: derive from tokenizer (include added tokens)
embed_dim = 64
num_classes = 2   # TODO: set to 2
learning_rate = 1e-3 # TODO: set to 1e-3
if None in (vocab_size, num_classes, learning_rate):
    raise ValueError('Set vocab_size, num_classes, and learning_rate before training.')


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
optimizer = torch.optim.Adam(attn_model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

attn_model.train()
for batch in train_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    optimizer.zero_grad()
    outputs = attn_model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

print('Custom Attention model trained on SMS dataset. Sample batch loss:', loss.item())

Custom Attention model trained on SMS dataset. Sample batch loss: 0.3084579110145569


# Partie 5: Métriques et Évaluation
Chargez la précision, le rappel et le F1 à partir de `evaluate`, puis implémentez la fonction d'aide `compute_metrics` partagée.

In [12]:
# Installation de secours si le module est manquant
%pip install --quiet evaluate

# TODO: configure evaluation metrics
import evaluate
import numpy as np

accuracy = evaluate.load('accuracy')   # TODO: 'accuracy'
precision = evaluate.load('precision')  # TODO: 'precision'
recall = evaluate.load('recall')     # TODO: 'recall'
f1 = evaluate.load('f1')         # TODO: 'f1'


def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy'],
        'precision': precision.compute(predictions=preds, references=labels)['precision'],
        'recall': recall.compute(predictions=preds, references=labels)['recall'],
        'f1': f1.compute(predictions=preds, references=labels)['f1'],
    }

> **Point d'apprentissage**
> Utilisez le même modèle de dictionnaire d'aide pour GPT-2 et le modèle personnalisé afin de pouvoir comparer les métriques côte à côte.

In [13]:
# TODO: evaluate GPT-2 on the validation split
gpt2_preds = []
gpt2_labels = []
model.to(device)
model.eval()
for ex in val_tok:
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(inputs).logits
    pred = torch.argmax(logits, dim=-1).cpu().item()
    gpt2_preds.append(pred)
    gpt2_labels.append(ex['label'])


gpt2_metrics = {
    'accuracy': accuracy.compute(predictions=gpt2_preds, references=gpt2_labels)['accuracy'],
    'precision': precision.compute(predictions=gpt2_preds, references=gpt2_labels)['precision'],
    'recall': recall.compute(predictions=gpt2_preds, references=gpt2_labels)['recall'],
    'f1': f1.compute(predictions=gpt2_preds, references=gpt2_labels)['f1'],
}
print('GPT-2 Metrics:', gpt2_metrics)

[transformers] We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (50256) is identical to the `bos_token_id` (50256), `eos_token_id` (50256), or the `sep_token_id` (None), and your input is not padded.


GPT-2 Metrics: {'accuracy': 0.859, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}


In [14]:
# TODO: evaluate the custom attention model
attn_preds = []
attn_labels = []
attn_model.eval()
for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
        preds = torch.argmax(outputs, dim=1)
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())


attn_metrics = {
    'accuracy': accuracy.compute(predictions=attn_preds, references=attn_labels)['accuracy'],
    'precision': precision.compute(predictions=attn_preds, references=attn_labels)['precision'],
    'recall': recall.compute(predictions=attn_preds, references=attn_labels)['recall'],
    'f1': f1.compute(predictions=attn_preds, references=attn_labels)['f1'],
}
print('Attention Model Metrics:', attn_metrics)

Attention Model Metrics: {'accuracy': 0.862, 'precision': 0.5555555555555556, 'recall': 0.03597122302158273, 'f1': 0.06756756756756757}


# Partie 6: Questions de Réflexion
Répondez directement dans les cellules markdown ci-dessous une fois vos expériences terminées.

### 1. Quels sont les rôles de la requête, de la clé et de la valeur dans le mécanisme d'attention ?
- **Requête (Query) :** Représente l'élément actuel qui cherche des informations (ce que je cherche).
- **Clé (Key) :** Sert d'étiquette d'index pour chaque élément de la séquence (comment je suis indexé).
- **Valeur (Value) :** Contient l'information réelle associée à chaque clé (ce que je fournis si je suis pertinent).
L'attention calcule un score de compatibilité entre la requête et les clés pour déterminer le poids à accorder à chaque valeur correspondante.

### 2. Pourquoi utilisons-nous un facteur d'échelle dans l'attention par produit scalaire ?
Nous utilisons $1/\sqrt{d_k}$ car, à mesure que la dimension d'immersion ($d_k$) augmente, la magnitude du produit scalaire tend à croître. Cela peut pousser la fonction softmax vers des régions où les gradients sont extrêmement faibles (problème de saturation), rendant l'entraînement difficile. Le facteur d'échelle stabilise les gradients.

### 3. En quoi l'auto-attention diffère-t-elle des modèles de séquence traditionnels comme les RNN ?
Contrairement aux RNN qui traitent les jetons séquentiellement (un par un), l'auto-attention permet un traitement **parallèle** de toute la séquence. Elle capture les dépendances à longue distance beaucoup plus efficacement car le chemin entre deux jetons est constant (O(1)), alors que dans un RNN, il est proportionnel à la distance entre eux.

### 4. Analyse des performances
- **Analyse :** Le modèle d'attention personnalisé affiche une précision de ~86%, mais un rappel très faible (~0.03), ce qui signifie qu'il détecte peu de spams mais se trompe peu quand il le fait. GPT-2, sans fine-tuning supplémentaire ici, prédit majoritairement la classe 'ham'.
- **Compromis :** GPT-2 est bien plus puissant mais lourd (548 Mo). Le modèle personnalisé est extrêmement léger et rapide mais manque de contexte pré-entraîné.
- **Amélioration :** On pourrait ajouter plusieurs têtes d'attention (Multi-Head Attention) ou utiliser des plongements de mots pré-entraînés (comme GloVe) pour améliorer le modèle personnalisé.